In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
import lightgbm as lgb

In [3]:
msl_new = pd.read_csv('../input_data/msl_new.csv')
print(len(msl_new))
msl_new.head()

16648


,Chromophore,Solvent,Quantum yield
0,O=C([O-])c1ccccc1-c1c2ccc(=O)cc-2oc2cc([O-])ccc12,O,0.950
1,O=C([O-])c1ccccc1C1=c2cc3c4c(c2Oc2c1cc1c5c2CCC...,CO,1.000
2,O=C([O-])c1ccccc1-c1c2cc(Br)c(=O)c(Br)c-2oc2c(...,O,0.200
3,O=C([O-])c1ccccc1-c1c2cc(I)c(=O)c(I)c-2oc2c(I)...,O,0.020
4,O=C([O-])c1c(Cl)c(Cl)c(Cl)c(Cl)c1-c1c2cc(I)c(=...,O,0.018


In [4]:
fgp_mols = torch.load('../torch_data/fingerprints.pt')
fgp_sols = torch.load('../torch_data/fingerprints_sol.pt')
print(len(fgp_mols), len(fgp_sols))
print(fgp_mols.shape, fgp_sols.shape)

C:\Users\kappa\AppData\Local\Temp\ipykernel_10764\42225485.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  fgp_mols = torch.load('../torch_data/fingerprints.pt')
C:\User

16648 16648
torch.Size([16648, 2048]) torch.Size([16648, 2048])


In [ ]:
X = torch.cat([fgp_mols, fgp_sols], dim=1)

print(X.shape)

y = torch.tensor(msl_new['Quantum yield'].values)
print(y.shape)

torch.Size([16648, 4096])
torch.Size([16648])


In [28]:
import numpy as np
from sklearn.model_selection import train_test_split

X_np = X.cpu().numpy()
y_np = y.cpu().numpy()

X_train, X_temp, y_train, y_temp = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(13318, 4096) (13318,)
(1665, 4096) (1665,)
(1665, 4096) (1665,)


In [8]:
import json
with open("../model_params/xgb_params.json", "r") as f:
    params = json.load(f)

In [30]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_val, y_val)]

xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_xgb)):.3f}")

test:
R Squared: 0.662
RMSE: 0.175


### Mol-only descriptors and fingerprints

In [11]:
X = np.load("../model_data/X_fgp.npy")
print(X.shape)
print(len(X))

y = np.load("../model_data/y_gfp.npy")
print(y.shape)
print(len(y))

(16648, 2777)
16648
(16648,)
16648


In [14]:
import numpy as np
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(13318, 2777) (13318,)
(1665, 2777) (1665,)
(1665, 2777) (1665,)


In [15]:
xgb_model = XGBRegressor(
    **params,
    n_estimators=2000,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_val, y_val)]

xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_xgb)):.3f}")

test:
R Squared: 0.722
RMSE: 0.159


### Full Descriptors and Fingerprints

In [16]:
desc_mols = torch.load("../torch_data/descriptors.pt")
desc_sols = torch.load("../torch_data/descriptors_sols.pt")

C:\Users\kappa\AppData\Local\Temp\ipykernel_10764\3135145622.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  desc_mols = torch.load("../torch_data/descriptors.pt")
C:\Us

In [17]:
vecs_mols = torch.load('../torch_data/mols_new.pt')
vecs_sols = torch.load('../torch_data/sols_new.pt')

C:\Users\kappa\AppData\Local\Temp\ipykernel_10764\4242151166.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vecs_mols = torch.load('../torch_data/mols_new.pt')
C:\Users

In [18]:
print(desc_mols.shape, desc_sols.shape)
print(vecs_mols.shape, vecs_sols.shape)

torch.Size([16648, 217]) torch.Size([16648, 217])
torch.Size([16648, 256]) torch.Size([16648, 256])


In [19]:
X = torch.cat([vecs_mols, vecs_sols, desc_mols, desc_sols, fgp_mols, fgp_sols], dim=1)

print(X.shape)

y = torch.tensor(msl_new['Quantum yield'].values)
print(y.shape)

torch.Size([16648, 5042])
torch.Size([16648])


In [20]:
X_np = X.cpu().numpy()
y_np = y.cpu().numpy()

X_train, X_temp, y_train, y_temp = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(13318, 5042) (13318,)
(1665, 5042) (1665,)
(1665, 5042) (1665,)


In [21]:
xgb_model = XGBRegressor(
    **params,
    n_estimators=2000,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_val, y_val)]

xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_xgb)):.3f}")

test:
R Squared: 0.714
RMSE: 0.161
